# AETHER STT — phase 1 (CTC branch) training notebook

Clones `aether-v3` from GitHub and trains the CTC-only pipeline: frozen
Mimi encoder → semantic codes → `AetherSpeech` transformer → CTC head, on
LibriSpeech.

**Run `notebooks/prepare_data.ipynb` first** (on a cheap T4 runtime) to
build the Mimi extraction cache and push it to Google Drive - this
notebook only restores that cache and trains, it does not do extraction
itself, so it can run entirely on a stronger GPU tier (A100/L4) without
burning expensive compute units on CPU-bound dataset prep.

Checkpoints/logs also go straight to Drive, and training auto-resumes from
the last checkpoint there if one exists - a dropped session doesn't lose
the compute units already spent.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_NAME = "aether-v3"

# Idempotent regardless of how many times this cell is re-run in the same
# kernel session: after the first run cwd is already inside the repo (from
# os.chdir below), so checking os.path.isdir("aether-v3") relative to cwd
# would look one level too deep and clone a second copy nested inside the
# first - repeatable indefinitely. Instead, explicitly handle "already
# standing inside the repo" as its own case.
cwd = Path.cwd()
if cwd.name == REPO_NAME and (cwd / ".git").is_dir():
    subprocess.run(["git", "pull"], check=True, cwd=cwd)
    repo_dir = cwd
    print("Already inside the repo, pulled.")
else:
    repo_dir = cwd / REPO_NAME
    if (repo_dir / ".git").is_dir():
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
        print("Pulled")
    elif repo_dir.exists():
        raise RuntimeError(
            f"{repo_dir} exists but isn't a git checkout (no .git/) - "
            "remove or rename it manually before re-running this cell."
        )
    else:
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
        print("Cloned")
    os.chdir(repo_dir)

print("cwd:", os.getcwd())

In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# Most GPU notebook images already ship a CUDA-matched torch build — don't
# clobber it. Only install if genuinely missing.
if importlib.util.find_spec("torch") is None:
    pip_install("torch")

# Training only reads the already-extracted cache (`datasets.load_from_disk`)
# and scores WER/CER - it never touches Mimi or raw audio, so none of
# transformers/torchaudio/soundfile/librosa are needed here (see
# prepare_data.ipynb for those).
pip_install(
    "datasets>=2.19,<4.0",
    "jiwer",
    "pyyaml",
    "numpy",
    "tqdm",
)

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("using device:", DEVICE)
assert DEVICE == "cuda", "No GPU visible on this VM - check the Colab session's accelerator."

## Google Drive cache

Mounts Drive, points checkpoints/logs (`train.output_dir`) there directly
(survives a dropped/recycled session), and restores the extraction cache
built by `prepare_data.ipynb` - no re-extraction happens here.

In [ ]:
from pathlib import Path

from aether_v3.config import load_config

real_config = load_config("configs/ctc_base.yaml")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/aether-v3")
except ImportError:
    DRIVE_ROOT = Path("drive_cache").resolve()
    print(f"Not running in Colab - falling back to local '{DRIVE_ROOT}' (no cross-session persistence).")

DRIVE_CACHE_DIR = DRIVE_ROOT / "data_cache" / Path(real_config.data.cache_dir).name
DRIVE_RUN_DIR = DRIVE_ROOT / "runs" / Path(real_config.train.output_dir).name

# Checkpoints/logs go straight to Drive - a dropped session shouldn't cost
# the training progress already paid for in GPU units.
real_config.train.output_dir = str(DRIVE_RUN_DIR)

# Resume from a previous session's progress instead of starting over.
resume_path = DRIVE_RUN_DIR / "last.pt"
if resume_path.exists():
    real_config.train.resume_from = str(resume_path)
    print(f"Found existing checkpoint at {resume_path}, will resume from it.")

print("Drive cache dir:", DRIVE_CACHE_DIR)
print("Drive run dir:  ", DRIVE_RUN_DIR)

## Restore the extraction cache

Pulls the cache `prepare_data.ipynb` already pushed to Drive. If it's not
there yet, this raises rather than silently falling back to extracting on
this (likely more expensive) GPU tier - go run that notebook first.

In [ ]:
from pathlib import Path

from aether_v3.data.cache_sync import CACHE_ROLES, hydrate_from_remote

restored = hydrate_from_remote(real_config.data.cache_dir, DRIVE_CACHE_DIR)
missing = [r for r in CACHE_ROLES if not (Path(real_config.data.cache_dir) / r).exists()]
if missing:
    raise RuntimeError(
        f"Cache role(s) {missing} not found locally or on Drive ({DRIVE_CACHE_DIR}). "
        "Run notebooks/prepare_data.ipynb first (on a cheap GPU, e.g. T4) to build "
        "and push the extraction cache."
    )
print(f"Restored from Drive: {restored or '(nothing to restore, already local)'}")

### Train

**Single GPU:** run the Python cell below.

**Multiple GPUs on this machine:** don't use the Python cell — use the shell cell instead (`torchrun` spawns its own processes; the training loop auto-detects its environment variables and switches to DDP with no code changes).

Checkpoints/logs write straight to Drive (`real_config.train.output_dir`, set above), and if a `last.pt` was already there from a previous session, training resumes from it automatically — no edits needed after a dropped session or a runtime-type switch.

In [ ]:
from aether_v3.training.train_ctc import run_training

run_training(real_config)

In [ ]:
# Multi-GPU alternative to the cell above — edit nproc_per_node, then run this
# cell instead of the plain `run_training(real_config)` call.
# NPROC = 4
# !torchrun --nproc_per_node={NPROC} -m aether_v3.training.train_ctc --config configs/ctc_base.yaml

## Monitor training

Re-run this cell any time (even from a second notebook while the cell above is still training) to see the latest loss/WER/CER curves from `log.jsonl`.

Watch WER/CER, not eval loss - CTC-infeasible examples (target longer than the Mimi frame count) get `zero_infinity`-clamped to ~0 loss regardless of how well the model is actually doing, so eval loss alone is misleading.

In [ ]:
import json

import matplotlib.pyplot as plt

with open(f"{real_config.train.output_dir}/log.jsonl") as f:
    rows = [json.loads(line) for line in f]

train_rows = [r for r in rows if "loss" in r and "eval_loss" not in r]
eval_rows = [r for r in rows if "eval_cer" in r]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([r["step"] for r in train_rows], [r["loss"] for r in train_rows])
axes[0].set_title("train loss")
axes[0].set_xlabel("step")

axes[1].plot([r["step"] for r in eval_rows], [r["eval_cer"] for r in eval_rows], label="CER")
axes[1].plot([r["step"] for r in eval_rows], [r["eval_wer"] for r in eval_rows], label="WER")
axes[1].set_title("dev CER / WER")
axes[1].set_xlabel("step")
axes[1].legend()
plt.show()

if eval_rows:
    best = min(eval_rows, key=lambda r: r["eval_cer"])
    print("best eval so far:", best)